# 01 — Gemma 4 12B: Prompt Development for Nepali ASR Benchmarking

**Model:** `google/gemma-4-12b-it`  
**Architecture:** Encoder-free multimodal (text + image + audio)  
**Audio support:** Native, up to 30s, 16 kHz mono, 25 tokens/sec  
**Class:** `AutoModelForImageTextToText`


## 0. Install Dependencies

> ⚠️ Gemma 4 requires `transformers >= 4.54` and the latest `accelerate`. Run this cell first and **restart the runtime** if prompted.


In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg])

# Core — pinned versions that work together for Gemma 4
pip("transformers>=4.54")
pip("accelerate>=1.6")

pip("bitsandbytes>=0.45")
pip("sentencepiece")
pip("protobuf")

# Audio processing
pip("librosa>=0.10")
pip("soundfile>=0.12")
pip("scipy")

# Metrics
pip("jiwer>=3.1")
pip("jsonlines")

# Data
pip("pandas")
pip("tqdm")

print("\n✅ All dependencies installed.")


## A. Experiment Configuration


In [ ]:
import os, json, torch, gc
from datetime import datetime

MODEL_ID   = "google/gemma-4-12b-it"
MODEL_REV  = "main"
QUANT      = "bfloat16"   # Gemma 4 12B fits in bf16 on T4/P100 with device_map=auto

experiment_config = {
    "model_id":       MODEL_ID,
    "model_revision": MODEL_REV,
    "quantization":   QUANT,
    "random_seed":    42,
    "audio_sr":       16000,
    "batch_size":     1,       # process one at a time for reliability
    "max_new_tokens": 256,
    "temperature":    0.0,
    "do_sample":      False,
    "timestamp":      datetime.now().isoformat(),
}

AUDIO_BASE = "/kaggle/input/datasets/panditaadarsh/llm-bechmarking-audio"
RESULTS_DIR = "/kaggle/working/results/prompt_dev/gemma4_12b"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(f"{RESULTS_DIR}/run_config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

print("Config saved →", RESULTS_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A")


## B. Model Loading


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

print(f"Loading {MODEL_ID} ...")

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    revision=MODEL_REV,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print("✅ Model loaded successfully.")
print(f"   dtype: {model.dtype}")
print(f"   device_map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'N/A'}")


## B.1 — Quick Sanity Check (one audio file)


In [ ]:
import glob

# Pick the first .wav from the clean set
audio_files = sorted(glob.glob(f"{AUDIO_BASE}/clean_nepali_200_flat/*.wav"))
if not audio_files:
    audio_files = sorted(glob.glob(f"{AUDIO_BASE}/clean_nepali_200_flat/*.mp3"))
print(f"Found {len(audio_files)} audio files in clean set")

test_audio_path = audio_files[0]
print(f"Testing with: {test_audio_path}")

# Build conversation — pass the FILE PATH, not a numpy array.
# The processor will load it internally.
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": test_audio_path},
            {"type": "text", "text": "Transcribe the following Nepali speech verbatim in Devanagari script. Output ONLY the transcription, nothing else."},
        ],
    },
]

# Use tokenize=True so the processor handles EVERYTHING end-to-end.
# Use load_audio_backend='soundfile' to bypass Kaggle's broken torchcodec.
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    processor_kwargs={"load_audio_backend": "librosa"},
).to(model.device)

print(f"input_ids shape: {inputs['input_ids'].shape}")
print(f"input_features present: {'input_features' in inputs}")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)

# Decode only newly generated tokens
new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
result = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
print(f"\n📝 Model output:\n{result}")


## C. Prompt Templates

Defining prompt levels 0-2 per `todo_bef_benchmark.md`.


In [ ]:
PROMPTS = {
    "L0_a": "Transcribe the following speech segment in its original language. Only output the transcription.",
    
    "L1_a": (
        "You are a speech transcription system. "
        "Transcribe the following Nepali audio into Nepali text using Devanagari script. "
        "Produce a verbatim transcription. Do not translate. "
        "Return only the transcription, nothing else."
    ),
    "L1_b": (
        "Task: verbatim Nepali speech transcription.\n"
        "Language: Nepali (Devanagari script).\n"
        "Instructions: transcribe exactly what is spoken. Do not translate. "
        "Output only the transcription."
    ),
    
    "L2_a": (
        "Transcribe the spoken Nepali audio verbatim in Devanagari script. "
        "Preserve any English words in Latin script. "
        "Maintain the order of Nepali–English code-switching as spoken. "
        "Do not translate between languages. "
        "Keep fillers, repetitions, corrections, and incomplete words. "
        "Do not correct grammar. Do not infer inaudible words. "
        "Do not add timestamps, speaker labels, explanations, or confidence scores. "
        "Return only the transcription."
    ),
    "L2_b": (
        "You are a verbatim transcription system for Nepali speech.\n"
        "Rules:\n"
        "1. Write Nepali in Devanagari.\n"
        "2. Write English words in Latin script.\n"
        "3. Preserve code-switching order.\n"
        "4. Do not translate.\n"
        "5. Keep fillers, repetitions, corrections, incomplete words.\n"
        "6. Do not correct grammar.\n"
        "7. Do not guess inaudible words.\n"
        "8. No timestamps, no speaker labels, no explanations.\n"
        "9. Output only the transcription."
    ),
}

print(f"Defined {len(PROMPTS)} prompt variants.")
for pid, text in PROMPTS.items():
    print(f"  {pid}: {text[:60]}...")


## D. Build Manifest


In [ ]:
import pandas as pd

def build_manifest(audio_dir, condition, max_files=None):
    '''Scan audio directory and build a manifest DataFrame, loading references from CSV if available.'''
    import glob, os
    
    # Try to load metadata
    metadata_df = None
    for meta_name in ["metadata.csv", "noisy_metadata.csv"]:
        meta_path = os.path.join(audio_dir, meta_name)
        if os.path.exists(meta_path):
            metadata_df = pd.read_csv(meta_path)
            # Ensure we have a consistent identifier to join on
            if "file" in metadata_df.columns:
                metadata_df["utterance_id"] = metadata_df["file"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])
            break
            
    files = sorted(glob.glob(f"{audio_dir}/**/*.wav", recursive=True)) +             sorted(glob.glob(f"{audio_dir}/**/*.mp3", recursive=True))
            
    if max_files:
        files = files[:max_files]
        
    records = []
    for fp in files:
        uid = os.path.splitext(os.path.basename(fp))[0]
        
        # Look up reference
        ref_text = ""
        if metadata_df is not None and "utterance_id" in metadata_df.columns:
            match = metadata_df[metadata_df["utterance_id"] == uid]
            if not match.empty:
                # Use label_normalized if available, else reference
                if "label_normalized" in match.columns:
                    ref_text = str(match.iloc[0]["label_normalized"])
                elif "reference" in match.columns:
                    ref_text = str(match.iloc[0]["reference"])
                    
        records.append({
            "utterance_id": uid,
            "audio_path": fp,
            "speech_condition": condition,
            "reference_raw": ref_text,
        })
    return pd.DataFrame(records)


manifest_clean = build_manifest(f"{AUDIO_BASE}/clean_nepali_200_flat", "clean", max_files=5)
manifest_noisy = build_manifest(f"{AUDIO_BASE}/noisy_nepali_200", "noisy", max_files=5)
manifest_cs    = build_manifest(f"{AUDIO_BASE}/codeswitched_nepali_200_flat", "codeswitched", max_files=5)

manifest = pd.concat([manifest_clean, manifest_noisy, manifest_cs], ignore_index=True)
print(f"Pilot manifest: {len(manifest)} utterances")
print(manifest.speech_condition.value_counts().to_dict())


## E. Batch Inference Pipeline


In [ ]:
import time, traceback
import jsonlines
from tqdm.auto import tqdm

def transcribe_one(audio_path, prompt_text):
    """Run inference for a single audio + prompt pair."""
    # Pass the FILE PATH directly — the processor loads audio internally.
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": audio_path},
                {"type": "text", "text": prompt_text},
            ],
        },
    ]
    
    # tokenize=True lets the processor handle everything end-to-end.
    # load_audio_backend='soundfile' bypasses Kaggle's broken torchcodec.
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={"load_audio_backend": "librosa"},
    ).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=experiment_config["max_new_tokens"],
            do_sample=experiment_config["do_sample"],
        )
    
    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0]


def clean_output(raw_output):
    """Basic output cleaning — remove markdown wrappers, labels, whitespace."""
    text = raw_output.strip()
    # Remove common wrappers
    for prefix in ["Transcription:", "Output:", "```", "**"]:
        if text.startswith(prefix):
            text = text[len(prefix):]
    text = text.strip("`*\n ")
    return text


def run_pipeline(manifest_df, prompts_dict, output_file):
    """Run batch inference with checkpointing."""
    output_path = f"{RESULTS_DIR}/{output_file}"
    
    # Load existing results for resume support
    completed = set()
    if os.path.exists(output_path):
        with jsonlines.open(output_path) as reader:
            for obj in reader:
                completed.add((obj["utterance_id"], obj["prompt_id"]))
        print(f"Resuming: {len(completed)} already completed.")
    
    total = len(manifest_df) * len(prompts_dict)
    pbar = tqdm(total=total, desc="Inference")
    
    for _, row in manifest_df.iterrows():
        for prompt_id, prompt_text in prompts_dict.items():
            key = (row["utterance_id"], prompt_id)
            if key in completed:
                pbar.update(1)
                continue
            
            record = {
                "model_id": MODEL_ID,
                "utterance_id": row["utterance_id"],
                "prompt_id": prompt_id,
                "prompt_level": prompt_id.split("_")[0],
                "prompt_variant": prompt_id.split("_")[1] if "_" in prompt_id else "a",
                "audio_path": row["audio_path"],
                "speech_condition": row["speech_condition"],
                "reference_raw": row.get("reference_raw", ""),
                "status": "success",
                "raw_output": "",
                "cleaned_prediction": "",
                "inference_seconds": 0,
                "timestamp": datetime.now().isoformat(),
            }
            
            try:
                t0 = time.time()
                raw = transcribe_one(row["audio_path"], prompt_text)
                record["inference_seconds"] = round(time.time() - t0, 2)
                record["raw_output"] = raw
                record["cleaned_prediction"] = clean_output(raw)
                
                if not record["cleaned_prediction"]:
                    record["status"] = "empty_output"
                    
            except torch.cuda.OutOfMemoryError:
                record["status"] = "out_of_memory"
                gc.collect()
                torch.cuda.empty_cache()
            except Exception as e:
                record["status"] = "inference_error"
                record["raw_output"] = str(e)
            
            # Append immediately for checkpoint
            with jsonlines.open(output_path, mode="a") as writer:
                writer.write(record)
            
            completed.add(key)
            pbar.update(1)
    
    pbar.close()
    print(f"\n✅ Pipeline complete. {len(completed)} results saved to {output_path}")
    return output_path


## F. Run Inference


In [ ]:
results_file = run_pipeline(manifest, PROMPTS, "raw_predictions.jsonl")


## G. Compute Metrics


In [ ]:
from jiwer import wer, cer

def compute_metrics(ref, hyp):
    """Compute WER and CER. Returns dict."""
    if not ref or not hyp:
        return {"wer": 1.0, "cer": 1.0}
    try:
        w = wer(ref, hyp)
        c = cer(ref, hyp)
    except Exception:
        w, c = 1.0, 1.0
    return {"wer": round(w, 4), "cer": round(c, 4)}

# Load results and compute metrics
results = []
with jsonlines.open(f"{RESULTS_DIR}/raw_predictions.jsonl") as reader:
    for obj in reader:
        if obj["status"] == "success" and obj.get("reference_raw"):
            m = compute_metrics(obj["reference_raw"], obj["cleaned_prediction"])
            obj.update(m)
        results.append(obj)

df = pd.DataFrame(results)
df.to_csv(f"{RESULTS_DIR}/utterance_metrics.csv", index=False)

# Prompt-level summary
if "wer" in df.columns:
    summary = df[df["status"]=="success"].groupby(["prompt_id"]).agg(
        avg_wer=("wer", "mean"),
        avg_cer=("cer", "mean"),
        count=("utterance_id", "count"),
    ).reset_index().sort_values("avg_wer")
    summary.to_csv(f"{RESULTS_DIR}/prompt_summary.csv", index=False)
    print("\n📊 Prompt Summary:")
    display(summary)
else:
    print("No reference transcriptions available — skipping WER/CER.")
    print("Status distribution:")
    print(df["status"].value_counts())
